# 00-baseline — 압축 없이 통과시키기

압축을 하지 않는 랩입니다. 두 가지를 위해 있습니다.

1. **기준선** — 다른 랩의 "30% 절감" 이 무엇 대비인지 정해 줍니다
2. **하네스 검증** — 압축을 안 했으니 절감은 0%, 보존율은 100% 여야 합니다.
   아니면 압축기가 아니라 **측정 도구가 고장 난 것**입니다

먼저 원리를 한 단계씩 보고, 마지막에 `configs/` 의 모든 조건을 돌립니다.

## 1. kit 불러오기

랩은 `labs/` 를 경로에 넣고 `from kit import ...` 로 씁니다.

In [ ]:
import sys
from pathlib import Path

LAB = Path.cwd().resolve()
LABS = LAB.parents[0]                  # labs/<이 랩> -> labs
sys.path.insert(0, str(LABS))
sys.path.insert(0, str(LAB))           # 이 랩의 모듈(transforms, blocks 등)

from kit import VERSION, config as C, dataset, env, metrics, tokens as T
from kit.display import table, pct
from kit.runner import Run

# .env 는 labs/.env → 저장소 루트 .env → scripts/explore/.env 순으로 찾습니다.
env.load(verbose=True)

RUNS = LABS.parent / "runs"
print("kit", VERSION, "· 랩", LAB.name)

print("배포명 기본값:", env.get("AZURE_OPENAI_DEPLOYMENT", "(없음)"))
print("엔드포인트    :", env.mask_endpoint(env.get("AZURE_OPENAI_ENDPOINT")))

## 2. 설정 읽기

설정은 코드가 아니라 yaml 에 둡니다. 그래야 `runs/` 경로에 조건 이름이
그대로 남아서, 나중에 무엇을 돌렸는지 알 수 있습니다.

In [ ]:
cfg = C.load("configs/noop.yaml")

table(
    ["항목", "값"],
    [["name", cfg.name], ["lab", cfg.lab],
     ["params", cfg.params or "(없음)"],
     ["dataset.path", Path(cfg.dataset["path"]).name],
     ["model", cfg.model],
     ["tokenizer", cfg.tokenizer or "(기본: local)"]],
    align=["left", "left"], title="설정",
)

## 3. 토큰을 어떻게 셀 것인가

**두 가지 방식이 있고 설정으로 고릅니다.**

| | `local` | `api` |
|---|---|---|
| 방법 | tiktoken (없으면 문자 근사) | 모델을 호출해 `usage.input_tokens` |
| 비용 | 0 | 텍스트마다 호출 1회 |
| 정확도 | 근사 | **과금 기준 그대로** |
| 포함되는 것 | 텍스트만 | 텍스트 + **메시지 포맷 오버헤드** |

```yaml
tokenizer:
  mode: api            # local | api
  deployment: gpt-5.4  # 생략하면 AZURE_OPENAI_DEPLOYMENT
  cache: true          # 같은 텍스트는 한 번만 호출
```

**`api` 는 캐시가 필수입니다.** 케이스 N건이면 압축 전후로 2N 회를 부르고,
스윕을 10단계 돌리면 그만큼 곱해집니다. 결과는 `kit/.cache/` 에 남습니다.

> **주의** — `api` 값에는 메시지 포맷 오버헤드가 포함됩니다(실측 +6).
> 압축 전후를 같은 방식으로 재므로 **비율 비교는 안전하지만**,
> `local` 값과 나란히 놓으면 안 됩니다. 그래서 `token_backend` 를 기록합니다.

In [ ]:
counter = T.make_counter(cfg.tokenizer, cfg.model)
print("측정 방식:", counter.backend)

sample = "환불 수수료는 결제금액의 10%입니다."
print(f"예시 {len(sample)}자 → {counter(sample):,} 토큰")

## 4. 코퍼스 살펴보기

`must_include` 는 **정답에 꼭 필요한 문자열**입니다. 이게 있어야 보존율을
잴 수 있고, LLM 호출이 없으므로 스윕을 수백 번 돌려도 비용이 0 입니다.

In [ ]:
cases = dataset.load(cfg.dataset["path"], limit=cfg.dataset.get("limit"))
info = dataset.summarize(cases)

table(
    ["항목", "값"],
    [["케이스", f'{info["n_cases"]}건'],
     ["총 길이", f'{info["n_chars"]:,}자'],
     ["must_include 보유", f'{info["with_must_include"]}건'],
     ["유형", ", ".join(f"{k}×{v}" for k, v in info["kinds"].items())]],
    align=["left", "left"], title="코퍼스",
)

c = cases[0]
print(f"\n[{c.id}] {c.kind}")
print(f"질문        : {c.question}")
print(f"must_include: {c.must_include}")
print(f"원문        : {c.text[:80]}…")

## 5. 압축 — 이 랩은 그대로 통과시킵니다

**모든 랩이 이 시그니처를 씁니다.** 랩을 바꾼다는 건 이 함수 하나를
바꾼다는 뜻이고, 나머지(코퍼스·지표·기록)는 전부 재사용됩니다.

In [ ]:
def compress(text: str, **params) -> tuple[str, dict]:
    """압축하지 않습니다. 반환은 (압축문, 메타) 입니다."""
    return text, {}


after, meta = compress(cases[0].text)
print("원문과 동일:", after == cases[0].text)

## 6. 집계 — 평균만 보면 안 됩니다

**정답 보존율**은 각 케이스에서 `must_include` 문자열 중 압축 후에도 남은
비율입니다. 100% 면 하나도 안 잃은 것입니다.

집계는 평균과 함께 **최저값**과 **유형별 분해**를 항상 냅니다.

> 평균 90% 여도 한 케이스가 0% 면 그 질문에는 아예 답할 수 없습니다.
> 평균은 그걸 가립니다. **최저 보존율부터 보세요.**

In [ ]:
records = [
    metrics.per_case(c.id, c.kind, c.text, compress(c.text, **cfg.params)[0],
                     c.must_include, counter)
    for c in cases
]
m = metrics.aggregate(records, counter)

table(
    ["지표", "값", "정상값", "뜻"],
    [["절감률", pct(m["saved"]), "0.0%", "압축을 안 했으므로"],
     ["토큰", f'{m["tokens_before"]:,} → {m["tokens_after"]:,}', "변화 없음", ""],
     ["평균 보존율", pct(m.get("survival_mean")), "100%", "전체 케이스 평균"],
     ["하위 5%", pct(m.get("survival_p5")), "100%", "나쁜 쪽 5% 지점"],
     ["최저 보존율", pct(m.get("survival_worst")), "100%", "가장 많이 깨진 케이스"],
     ["측정 방식", m["token_backend"], "local 또는 api", "다르면 비교 불가"]],
    align=["left", "right", "right", "left"], title="집계",
)

## 7. 하네스 자가 점검

압축을 안 했으니 아래가 성립해야 합니다. 어긋나면 압축기가 아니라
**측정 도구가 고장 난 것**이고, 그 상태로는 다른 랩의 숫자를 믿을 수 없습니다.

In [ ]:
problems = []
if m["saved"] != 0.0:
    problems.append(f'절감률이 0 이 아닙니다 ({m["saved"]:.2%}) — 로더가 원문을 바꾸고 있습니다')
if m.get("survival_worst") not in (None, 1.0):
    problems.append(f'최저 보존율이 100% 가 아닙니다 ({m["survival_worst"]:.1%})')

if problems:
    print("하네스 점검 실패")
    for p in problems:
        print("  ✗", p)
else:
    print("하네스 정상 — 다른 랩을 돌려도 됩니다.")

## 8. 이 랩의 모든 조건 돌려보기

**조건 1개 = 파일 1개**입니다. `configs/` 를 훑으면 이 랩이 답할 수 있는
질문이 전부 나옵니다. 설정을 새로 추가해도 이 셀은 고칠 필요가 없습니다.

각 조건은 `runs/00-baseline/<설정이름>/<시각>/` 에 따로 기록됩니다. 나중에
"그때 무엇을 돌렸나" 를 설정 이름만 보고 알 수 있게 하려는 것입니다.

이 랩에는 조건이 둘 있습니다.

| 설정 | 토큰 측정 | 비용 |
|---|---|---|
| `noop.yaml` | `local` (tiktoken) | 0 |
| `noop-api.yaml` | `api` (모델 호출 실측) | 텍스트당 1회, 캐시되면 0 |

`api` 조건은 `.env` 가 없으면 건너뜁니다. 건너뛴 이유를 표에 남겨서,
"돌았는데 결과가 없는" 상태와 "아예 못 돌린" 상태를 구분합니다.

In [ ]:
def run_config(path):
    """설정 하나를 끝까지 돌리고 (설정, 지표, 결과경로) 를 돌려줍니다."""
    cfg = C.load(path)
    cases = dataset.load(cfg.dataset["path"], limit=cfg.dataset.get("limit"))
    counter = T.make_counter(cfg.tokenizer, cfg.model)

    run = Run(cfg, RUNS)
    for c in cases:
        after, extra = compress(c.text, **cfg.params)
        run.add(metrics.per_case(c.id, c.kind, c.text, after,
                                 c.must_include, counter, extra),
                before=c.text, after=after)

    m = metrics.aggregate(run.records, counter)
    m["dataset_name"] = Path(cfg.dataset["path"]).name
    counter.save()
    if counter.stats():
        m["token_calls"] = counter.stats()
    return cfg, m, run.finish(m, ["압축 없음. 다른 랩의 절감률은 이 결과를 기준으로 읽습니다."])


results, skipped = [], []
for p in sorted(Path("configs").glob("*.yaml")):
    try:
        cfg, m, out = run_config(p)
        results.append((cfg.name, m, out))
        extra = ""
        if m.get("token_calls"):
            extra = (f' · API {m["token_calls"]["api_calls"]}회'
                     f' · 캐시 {m["token_calls"]["cache_hits"]}회')
        print(f'{p.name:20s} 절감 {m["saved"]:6.1%} · {m["token_backend"]}{extra}')
    except Exception as e:
        skipped.append((p.name, f"{type(e).__name__}: {e}"))
        print(f"{p.name:20s} 건너뜀 — {type(e).__name__}: {str(e)[:80]}")

if skipped:
    print("\n건너뛴 설정이 있습니다. 자격증명이 없으면 api 조건은 못 돌립니다.")
    print("  cd labs && cp .env.example .env   (또는 cp ../scripts/explore/.env .env)")

## 9. 조건 비교

같은 코드에 조건만 바꿔 돌린 결과입니다. **숫자 하나가 아니라 표를 보세요.**
어떤 조건에서 무엇을 얻고 무엇을 잃는지가 이 랩의 결론입니다.

`local` 과 `api` 의 토큰 수가 다른 것이 정상입니다. 차이는 **메시지 포맷
오버헤드**(케이스당 +6)이고, 이건 텍스트가 아니라 역할 구분자 같은
프레이밍입니다. 그래서 두 값을 나란히 놓고 "어느 쪽이 맞다" 를 따지면 안 됩니다.

In [ ]:
table(
    ["설정", "측정 방식", "건수", "토큰", "절감", "최저 보존율"],
    [[n, m["token_backend"], m["n"],
      f'{m["tokens_before"]:,} → {m["tokens_after"]:,}',
      pct(m["saved"]), pct(m.get("survival_worst"))]
     for n, m, _ in results],
    align=["left", "left", "right", "right", "right", "right"],
    title="조건 비교",
    note="절감 0% · 최저 보존율 100% 가 두 조건 모두에서 나와야 하네스가 정상입니다.",
)

if len(results) == 2:
    a, b = (m for _, m, _ in results)
    diff = abs(a["tokens_before"] - b["tokens_before"])
    print(f"측정 방식 차이: {diff:,} 토큰 ({diff / max(a["n"], 1):.1f}/건)")
    print("케이스당 +6 이면 메시지 포맷 오버헤드입니다 — 텍스트가 아니라 프레이밍입니다.")

## 정리

- **기준선이 없으면 절감률은 의미가 없습니다** — 무엇 대비인지가 있어야 합니다
- **하네스를 먼저 검증합니다** — `kit` 을 고친 뒤에는 항상 여기부터 돌리세요
- **정답 보존율은 평균 대신 최저값** — 평균은 한 케이스의 붕괴를 가립니다
- **측정 방식을 기록합니다** — `local` 과 `api` 는 값이 다르므로 섞으면 안 됩니다

### 다음 랩

[`01-lossless-structure`](../01-lossless-structure/run.ipynb) — 의미를 하나도
안 버리고 표현만 바꿉니다. `compress()` 만 바뀌고 나머지는 그대로입니다.